# Tennessee Data Center Interactive Map Generator

This notebook generates the interactive Tennessee data-center map from the deduplicated `Master` sheet in the data file.

The workflow is:

**Data file → cleaned records → JSON → HTML template → interactive map**

The page structure, styles, controls, and JavaScript are stored in a separate template file. Future interface changes should be made in `template.html` rather than in this notebook.

The template must contain the placeholder:

`__DATA_JSON__`

The notebook replaces that placeholder with the current facility records.

## 1. Configuration

Set the data path, template path, source sheet, state abbreviation, output path, and preview option here.

Required packages:
pandas, numpy, openpyxl, IPython

In [1]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
from IPython.display import IFrame, display

DATA_FILE = Path("../dataset/tennessee_public_data_centers.xlsx")
TEMPLATE_FILE = Path("template/TN_dcmap_template.html")
DATA_SHEET = "Master"
CANDIDATE_SHEET = "Candidate_Sites"
STATE_ABBR = "TN"
OUTPUT_HTML = Path("tennessee_dcmap.html")
PREVIEW_IN_NOTEBOOK = True


## 2. JSON-safe value conversion

These helper functions normalize pandas and NumPy values before serialization and convert pipe-separated evidence URLs into a list.

In [2]:
def clean_value(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value) if math.isfinite(float(value)) else None
    if isinstance(value, pd.Timestamp):
        return value.strftime("%Y-%m-%d")
    return str(value)


def parse_evidence_urls(value):
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except Exception:
        pass
    return [
        item.strip()
        for item in str(value).split("|")
        if item.strip().startswith(("http://", "https://"))
    ]

## 3. Load and normalize the data

The notebook loads both the deduplicated `Master` sheet and the separate `Candidate_Sites` sheet.

`Master` remains the canonical confirmed-facility interface. Candidate records are serialized separately so they never alter the controlled lifecycle vocabulary or confirmed-facility counts.


In [3]:
data = pd.read_excel(DATA_FILE, sheet_name=DATA_SHEET)
candidates = pd.read_excel(DATA_FILE, sheet_name=CANDIDATE_SHEET)

required_columns = {
    "facility_id", "facility_name", "operator", "state", "latitude", "longitude",
    "facility_type", "analysis_scope", "status_normalized",
}
missing = required_columns - set(data.columns)
if missing:
    raise KeyError(sorted(missing))

candidate_required = {"candidate_id", "candidate_site_name", "candidate_type", "candidate_confidence", "state"}
candidate_missing = candidate_required - set(candidates.columns)
if candidate_missing:
    raise KeyError(sorted(candidate_missing))

data = data[data["state"].astype(str).str.upper().eq(STATE_ABBR)].copy()
data["latitude"] = pd.to_numeric(data["latitude"], errors="coerce")
data["longitude"] = pd.to_numeric(data["longitude"], errors="coerce")

for col in ["capacity_mw", "square_feet", "property_acres", "investment_usd", "source_count",
            "independent_evidence_domain_count", "source_confidence_score", "coordinate_max_spread_km"]:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

if "capacity_mw" not in data.columns:
    data["capacity_mw"] = np.nan

data = data.dropna(subset=["latitude", "longitude"]).copy()

candidates = candidates[candidates["state"].astype(str).str.upper().eq(STATE_ABBR)].copy()
candidates["latitude"] = pd.to_numeric(candidates.get("latitude"), errors="coerce")
candidates["longitude"] = pd.to_numeric(candidates.get("longitude"), errors="coerce")

print(f"Master map records: {len(data)}")
print(f"Candidate records: {len(candidates)}; mappable candidates: {candidates[['latitude','longitude']].notna().all(axis=1).sum()}")


Master map records: 45
Candidate records: 4; mappable candidates: 3


## 4. Build the facility records

Each row is converted to the schema consumed by the map template. Optional values remain `null` when unavailable.

In [4]:
def clean_bool(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {"true", "1", "yes"}:
        return True
    if text in {"false", "0", "no"}:
        return False
    return None


def numeric_or_none(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
        n = float(value)
        return n if math.isfinite(n) else None
    except (TypeError, ValueError):
        return None


records = []
for _, row in data.iterrows():
    records.append({
        "facility_id": clean_value(row.get("facility_id")),
        "facility_name": clean_value(row.get("facility_name")),
        "operator": clean_value(row.get("operator")) or "Unknown",
        "address": clean_value(row.get("address")),
        "city": clean_value(row.get("city")),
        "county": clean_value(row.get("county")),
        "state": clean_value(row.get("state")),
        "postal_code": clean_value(row.get("postal_code")),
        "lat": float(row["latitude"]),
        "lon": float(row["longitude"]),
        "location_precision": clean_value(row.get("location_precision")),
        "location_confidence": clean_value(row.get("location_confidence")),
        "coordinate_max_spread_km": numeric_or_none(row.get("coordinate_max_spread_km")),
        "coordinate_conflict_flag": clean_bool(row.get("coordinate_conflict_flag")),
        "status": clean_value(row.get("status_normalized")) or "unknown",
        "status_conflict_flag": clean_bool(row.get("status_conflict_flag")),
        "facility_type": clean_value(row.get("facility_type")) or "other",
        "analysis_scope": clean_value(row.get("analysis_scope")),
        "capacity_mw": numeric_or_none(row.get("capacity_mw")),
        "square_feet": numeric_or_none(row.get("square_feet")),
        "property_acres": numeric_or_none(row.get("property_acres")),
        "investment_usd": numeric_or_none(row.get("investment_usd")),
        "sources": clean_value(row.get("sources")),
        "source_count": numeric_or_none(row.get("source_count")),
        "last_updated": clean_value(row.get("last_updated")),
        "verification_class": clean_value(row.get("verification_class")),
        "operator_status": clean_value(row.get("operator_status")),
        "independent_evidence_domain_count": numeric_or_none(row.get("independent_evidence_domain_count")),
        "cross_verified": clean_bool(row.get("cross_verified")),
        "source_confidence_score": numeric_or_none(row.get("source_confidence_score")),
        "source_confidence_class": clean_value(row.get("source_confidence_class")),
        "verification_note": clean_value(row.get("verification_note")),
        "evidence_urls": parse_evidence_urls(row.get("evidence_urls")),
    })

candidate_records = []
for _, row in candidates.iterrows():
    lat = numeric_or_none(row.get("latitude"))
    lon = numeric_or_none(row.get("longitude"))
    candidate_records.append({
        "candidate_id": clean_value(row.get("candidate_id")),
        "master_facility_id": clean_value(row.get("master_facility_id")),
        "candidate_site_name": clean_value(row.get("candidate_site_name")),
        "candidate_type": clean_value(row.get("candidate_type")),
        "candidate_confidence": clean_value(row.get("candidate_confidence")),
        "address": clean_value(row.get("address")),
        "city": clean_value(row.get("city")),
        "county": clean_value(row.get("county")),
        "state": clean_value(row.get("state")),
        "postal_code": clean_value(row.get("postal_code")),
        "lat": lat,
        "lon": lon,
        "evidence_basis": clean_value(row.get("evidence_basis")),
        "source": clean_value(row.get("source")),
        "evidence_urls": parse_evidence_urls(row.get("evidence_urls")),
        "last_verified": clean_value(row.get("last_verified")),
        "review_note": clean_value(row.get("review_note")),
    })

data_json = json.dumps(records, ensure_ascii=False, separators=(",", ":"))
candidate_json = json.dumps(candidate_records, ensure_ascii=False, separators=(",", ":"))

len(records), len(candidate_records), sum(r["lat"] is not None and r["lon"] is not None for r in candidate_records)


(45, 4, 3)

## 5. Load the HTML template

The complete page implementation is read from `TEMPLATE_FILE`. The notebook does not contain the HTML, CSS, or JavaScript source.

In [5]:
html_template = TEMPLATE_FILE.read_text(encoding="utf-8")

for placeholder in ["__DATA_JSON__", "__CANDIDATE_JSON__"]:
    if placeholder not in html_template:
        raise ValueError(f"Missing template placeholder: {placeholder}")


## 6. Generate the interactive map

The facility JSON replaces the template placeholder and the final HTML is written to `OUTPUT_HTML`.

In [6]:
final_html = html_template.replace("__DATA_JSON__", data_json, 1)
final_html = final_html.replace("__CANDIDATE_JSON__", candidate_json, 1)
OUTPUT_HTML.write_text(final_html, encoding="utf-8")


85757

## 7. Preview

If `PREVIEW_IN_NOTEBOOK` is enabled, the generated HTML is displayed directly in the notebook.

In [7]:
if PREVIEW_IN_NOTEBOOK:
    display(IFrame(src=str(OUTPUT_HTML), width="100%", height=720))

## Updating the project

For confirmed-facility data changes, update `Master` and rerun the notebook.

For candidate-site updates, update `Candidate_Sites`; candidates remain a separate map layer and do not alter `Master` lifecycle semantics.

The expanded template supports verification/source-confidence/QA filtering and richer research-quality popups. Candidate rows without defensible coordinates remain visible in the candidate list but are intentionally not plotted until supported coordinates are added.
